# Demand Estimation and Market Analysis: Bluetooth Speakers

In this lab, you will study the market for Bluetooth speakers using brand-year data aggregated from Amazon purchases. The goal is to move from descriptive analysis to a simple demand model, and then use that model to infer markups and unit costs.

Use the adapted file:

```python
bluetooth_speakers_brand_year.csv
```

This file is adapted from `demand_4.parquet`. It keeps the five lecture brands from 2015-2023 and normalizes `brand_share` within those included brands so the notebook structure matches the air-fryer solutions. The original lecture-provided strategy outputs are also kept in `provided_...` columns so you can compare your estimates to the lecture benchmarks.


## Data

Each row is one brand in one year.

Important columns:

- `year`: calendar year
- `brand`: Bluetooth-speaker brand
- `purchase_count`: brand-year demand, adapted from the original `chosen_users` column
- `product_count`: set to 1 in this adapted file because the parquet file is already aggregated to the brand-year level
- `avg_price`: average price for that brand-year
- `avg_rating`: average review rating for that brand-year
- `waterproof_share`, `party_share`, `voice_assistant_share`: product-characteristic shares for that brand-year
- `brand_share`: purchase share within the included Bluetooth-speaker brands in that year
- `log_brand_share`: `np.log(brand_share)`, already computed for convenience
- `raw_brand_share`: the original `share_among_speaker_buyers` from the lecture setup
- `provided_dprime`, `provided_markup`, `provided_marginal_cost`, `provided_profit`: lecture-provided speaker-market fundamentals, included for comparison

This adapted speaker file restores the original Bluetooth-speaker product characteristics from the lecture notebook while keeping the notebook structure aligned with the air-fryer solutions.

As in the cleaned air-fryer solutions, we use:

$$
\log(s_{bt})
$$

as the outcome and include **year dummies**.


In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

## 1. Data Analysis

Load `bluetooth_speakers_brand_year.csv`.

1. Verify that the data contain 5 brands and the years 2015-2023.
2. Check that `brand_share` sums to 1 within each year.
3. Plot the following over time by brand:
   - average price
   - average rating
   - brand market share
4. Summarize the Bluetooth-speaker product characteristics:
   - Which brands are the most waterproof?
   - Which brands have the largest party-mode share?
   - Which brands have the largest voice-assistant share?
5. Describe the market in words. Which brands are expensive? Which brands have large shares? Which brands have the highest ratings? Does the market look stable over time?
6. Compare `brand_share` and `raw_brand_share`. What changed when we normalized shares within the included brands?

This part of the work is the **data analyst** role: making the data trustworthy, visual, and interpretable before building a model.


In [2]:
df = pd.read_csv("bluetooth_speakers_brand_year.csv")

print(df.shape)
print(df.head())
print(df.groupby("year")["brand_share"].sum())

feature_cols = [
    "waterproof_share",
    "party_share",
    "voice_assistant_share",]

(44, 25)
             category  year  brand  n_users  unique_users  purchase_count  \
0  bluetooth_speakers  2015  Anker       79            79              79   
1  bluetooth_speakers  2015   Bose     4212          4212            4212   
2  bluetooth_speakers  2015    JBL     2832          2832            2832   
3  bluetooth_speakers  2015  OontZ      104           104             104   
4  bluetooth_speakers  2016  Anker     2982          2982            2982   

   product_count   avg_price  avg_rating  waterproof_share  ...  user_share  \
0            1.0   63.329873    4.529114          0.000000  ...    0.003944   
1            1.0  141.189855    4.533974          0.000000  ...    0.210295   
2            1.0   78.035332    4.383510          0.000353  ...    0.141395   
3            1.0   45.456538    3.659615          0.000000  ...    0.005192   
4            1.0   37.254859    4.679376          0.000000  ...    0.112831   

   brand_share  log_brand_share  raw_brand_share  pro

## 2. Demand Estimation

We will estimate a logit-style demand model using linear regression. The model is:

$$
\log(s_{bt}) = \alpha_0 + \alpha_t + \gamma_b + \beta_{price}p_{bt} + \beta_{rating}r_{bt} + \sum_{\ell=1}^L \beta_\ell x_{bt\ell} + \epsilon_{bt}.
$$

Here:

- $b$ indexes brands
- $t$ indexes years
- $s_{bt}$ is `brand_share`
- $p_{bt}$ is `avg_price`
- $r_{bt}$ is `avg_rating`
- $x_{bt\ell}$ are `waterproof_share`, `party_share`, and `voice_assistant_share`
- $\alpha_t$ are year dummy coefficients
- $\gamma_b$ are brand dummy coefficients
- $\beta_{price}$ is **one constant price coefficient**, shared by all brands and all years

Use `pd.get_dummies(..., drop_first=True)` for brand and year dummies. The dropped brand and dropped year become the reference categories, so all dummy coefficients are interpreted relative to those omitted categories.

Questions:

1. What is the estimated price coefficient, $\hat{\beta}_{price}$?
2. Is it negative? Why is that important?
3. What is the coefficient on average rating?
4. Which feature coefficients are positive, and which are negative?
5. Which brand dummy coefficients are largest? Remember that these are interpreted relative to the dropped brand.
6. Which year dummy coefficients are largest? Remember that these are interpreted relative to the dropped year.
7. What is the model's $R^2$?

This part of the work is the **data scientist** role: turning the cleaned data into a model that can be used for prediction and interpretation.


In [3]:
y = df["log_brand_share"]

brand_dummies = pd.get_dummies(df["brand"], 
                               prefix="brand", drop_first=True, dtype=int)
year_dummies = pd.get_dummies(df["year"].astype(str), 
                              prefix="year", drop_first=True, dtype=int)

X = pd.concat(
    [df[["avg_price", "avg_rating"] + feature_cols],
    brand_dummies,
    year_dummies],
    axis=1,)

model = LinearRegression()
model.fit(X, y)

predicted_log_share = model.predict(X)
r2 = r2_score(y, predicted_log_share)

coef_table = pd.DataFrame({
    "feature": X.columns,
    "coefficient": model.coef_})

print("R-squared:", r2)
coef_table

R-squared: 0.8014823966204588


,feature,coefficient
0,avg_price,-0.060613
1,avg_rating,9.036807
2,waterproof_share,-3.839633
3,party_share,9.475497
4,voice_assistant_share,4.030405
5,brand_Bose,7.896362
6,brand_DOSS,1.745747
7,brand_JBL,6.291683
8,brand_OontZ,3.499626
9,year_2016,-0.524593


## 3. Strategy: Costs, Markups, and Profit

Now use the demand estimate to infer market fundamentals.

The price coefficient is constant across brands and years:

$$
\hat{\beta}_{price}.
$$

For each brand-year, compute the slope of demand with respect to price as:

$$
\hat{s}'_{bt}(p_{bt}) = \hat{\beta}_{price} s_{bt}(1 - s_{bt}).
$$

Then estimate unit cost, or marginal cost, using the firm's first-order pricing condition:

$$
\hat{c}_{bt} = p_{bt} + \dfrac{s_{bt}}{\hat{s}'_{bt}(p_{bt})}.
$$

Because $\hat{\beta}_{price}$ should be negative, $\hat{s}'_{bt}(p_{bt})$ should also be negative. If your price coefficient is positive, stop and debug your model before interpreting costs.

Compute:

- `demand_slope`: $\hat{s}'_{bt}(p_{bt})$
- `unit_cost`: $\hat{c}_{bt}$
- `markup`: $m_{bt} = p_{bt} - \hat{c}_{bt}$
- `average_profit`: $s_{bt} 	imes m_{bt}$

Here `average_profit` is a share-weighted profit index, not total dollars of profit. It is useful for comparing brand-years inside this adapted speaker market.

Questions:

1. What are the average unit costs and markups for each brand over the years?
2. Are any inferred unit costs negative? If so, what might that mean?
3. Which brands have the highest average unit costs? How do average unit cost and average ratings compare?
4. Make kernel density plots of unit costs, markups, and average profit.
5. Which brands have the highest share-weighted average profit?
6. Compare your estimated `markup`, `unit_cost`, and `average_profit` to the lecture-provided `provided_markup`, `provided_marginal_cost`, and `provided_profit`. How close are they?

This part of the work is the **pricing analyst** or **applied economist** role: using a demand model to reason about price, cost, and profitability.


In [4]:
price_coef = coef_table.loc[coef_table["feature"] == "avg_price", "coefficient"].iloc[0]
print("Estimated price coefficient:", price_coef)

results = df.copy()
results["predicted_log_share"] = predicted_log_share
results["demand_slope"] = price_coef * results["brand_share"] * (1 - results["brand_share"])
results["unit_cost"] = results["avg_price"] + results["brand_share"] / results["demand_slope"]
results["markup"] = results["avg_price"] - results["unit_cost"]
results["average_profit"] = results["brand_share"] * results["markup"]
results["profit_derivative"] = (
    results["demand_slope"] * results["markup"] + results["brand_share"]
)

print(results[["unit_cost", "markup", "average_profit", "profit_derivative"]].describe())

Estimated price coefficient: -0.06061346604245834
        unit_cost     markup  average_profit  profit_derivative
count   44.000000  44.000000       44.000000       4.400000e+01
mean    64.897758  21.561097        5.063113       2.463788e-18
std     44.918176   4.655795        4.655795       3.768830e-17
min     10.964197  16.499211        0.001227      -1.110223e-16
25%     29.760637  17.653303        1.155318       0.000000e+00
50%     40.225670  19.539376        3.041392       0.000000e+00
75%    104.428804  24.724874        8.226890       0.000000e+00
max    162.129221  39.545914       23.047930       1.110223e-16


In [5]:
results[[
    'brand',
    'markup',
    'provided_markup',
    'unit_cost',
    'provided_marginal_cost',
    'average_profit',
    'provided_profit',
]].groupby('brand').mean()


,markup,provided_markup,unit_cost,provided_marginal_cost,average_profit,provided_profit
brand,,,,,,
Anker,23.622763,19.295045,29.158654,33.486373,7.124779,102551.826438
Bose,22.579690,18.233182,133.300035,137.646543,6.081706,61673.579737
DOSS,17.439233,16.724884,34.137705,34.852054,0.941249,8960.429482
JBL,25.763496,19.791612,91.209634,97.181518,9.265512,111126.572110
OontZ,17.942319,17.133317,33.264980,34.073982,1.444335,26778.602252
